In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import os

In [2]:
CURRENT_DIRECTORY =  os.getcwd()
SRC_DIRECTORY = Path(CURRENT_DIRECTORY).parents[1]
print(SRC_DIRECTORY)

BASE_DATASET_PATH = Path(SRC_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/src
/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets


In [7]:
# get subzone data with their corresponding areas in km^2
def get_subzone_data(year: int):
    """
    Args
    ------
    year: int
        year of the subzones you want (2019, 2014, 2008)

    Returns
    ------
    Subzone dataframe containing the subzone names, planning area names and areas of points of interest.
    """

    year = str(year)

    subzones_filepath = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_{year}/subzone_boundary_{year}.gpkg")
    subzones_gpd = gpd.read_file(subzones_filepath)
    subzones_gpd = subzones_gpd.map(lambda s: s.lower() if isinstance(s, str) else s)
    # lower() the column names of subzones_gpd
    subzones_gpd.columns = subzones_gpd.columns.str.lower()

    # there is no landuse data found for 2008 masterplan, 
    # so not able to merge with the subzone_classification file obtained from extracting_landuse_type.ipynb
    if year == "2008":
        return subzones_gpd

    subzones_filepath = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_{year}/subzone_classifications_{year}.csv")
    subzones_csv = pd.read_csv(subzones_filepath)
    subzones_csv = subzones_csv.map(lambda s: s.lower() if isinstance(s, str) else s)

    subzones_with_geom = subzones_csv.merge(
        subzones_gpd[["subzone_n", "geometry"]],
        on = "subzone_n",
        how = "left"
    )

    missing_count = subzones_with_geom[subzones_with_geom.columns[-1]].isna().sum()
    if missing_count > 0:
        print(f"Warning: {missing_count} subzones did not find a match in the CSV.")

    return subzones_with_geom

def convert_to_geodataframe(df):
    if not isinstance(df, gpd.GeoDataFrame):
        df = gpd.GeoDataFrame(df, geometry='geometry')

    # ensure the original data has the correct starting CRS (WGS84)
    # (Only needed if it isn't already set; usually .gpkg files have this metadata)
    if df.crs is None:
        df.set_crs(epsg=4326, inplace=True)

    # transform the entire GeoDataFrame to SVY21 (Meters)
    df_meters = df.to_crs(epsg=3414)

    return df_meters

In [4]:
def get_education_data(year: int):

    year = str(year)

    education_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/education_combined.xlsx")
    education_df = pd.read_excel(education_filepath, sheet_name = f"{year}")

    education_df = education_df.rename(columns = {
        "planning_area": "pln_area_n"
    })

    return education_df

In [8]:
subzone_df = get_subzone_data(2008)
subzone_df = convert_to_geodataframe(subzone_df)
subzone_df

,objectid,subzone_no,subzone_n,subzone_c,ca_ind,pln_area_n,pln_area_c,region_n,region_c,inc_crc,fmel_upd_d,x_addr,y_addr,shape_leng,shape_area,geometry
0,1,1,woodlands regional centre,wdsz01,n,woodlands,wd,north region,nr,8877712517deec59,2016-05-11,22571.2094,46698.8735,3254.343275,5.956521e+05,"MULTIPOLYGON (((22629.486 46980.246, 22683.012..."
1,2,1,kranji,sksz01,n,sungei kadut,sk,north region,nr,de75f42ec0f86b2b,2016-05-11,19767.5129,46226.3625,7906.523525,3.654121e+06,"MULTIPOLYGON (((20329.947 47234.988, 20333.366..."
2,3,2,turf club,sksz02,n,sungei kadut,sk,north region,nr,5d555db858522781,2016-05-11,20234.1723,44507.1299,7677.244752,3.290816e+06,"MULTIPOLYGON (((21094.367 44815.121, 21099.854..."
3,4,7,nee soon,yssz07,n,yishun,ys,north region,nr,d5436ffe80750d15,2016-05-11,25922.1222,43260.2558,7074.178844,2.209311e+06,"MULTIPOLYGON (((26860.555 43913.328, 26839.719..."
4,5,9,yishun west,yssz09,n,yishun,ys,north region,nr,6cc6926ff0495d89,2016-05-11,27743.2683,45873.5516,5313.554643,1.517705e+06,"MULTIPOLYGON (((28228.609 45792.488, 28223.039..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
306,307,2,sungei serangoon west,sesz02,n,sengkang,se,north-east region,ner,dd1f0c466afa1e91,2016-05-11,35978.7574,41060.2561,6305.618929,1.572812e+06,"MULTIPOLYGON (((37009.969 40867.361, 37011.087..."
307,308,7,sungei serangoon,hgsz07,n,hougang,hg,north-east region,ner,fa12e3e0162969b6,2016-05-11,35870.2559,39071.4374,5788.601229,1.333531e+06,"MULTIPOLYGON (((36538.391 39761.352, 36517.439..."
308,309,1,hougang central,hgsz01,n,hougang,hg,north-east region,ner,3e946e53a7bafde2,2016-05-11,35042.8392,39344.8093,6868.798253,2.291220e+06,"MULTIPOLYGON (((35948.803 40264.817, 35951.927..."
309,310,3,changi west,chsz03,n,changi,ch,east region,er,64b23a491470d534,2016-05-11,44092.2139,38928.4138,14919.112423,4.849019e+06,"MULTIPOLYGON (((45347.86 41116.219, 45350.666 ..."


In [5]:
education_df = get_education_data(2010)
education_df

,pln_area_n,total,no_qualification,pri,lower_sec,sec,post_sec_(non-tertiary),poly_diploma,professional_qualification_&_other_diploma,uni
0,total,2779524,424443,193181,282523,526359,307562,250213,161144,634098
1,ang mo kio,140471,28602,11235,15798,24569,13592,11731,7162,27783
2,bedok,222290,33901,14542,22563,41999,24186,17438,12986,54675
3,bishan,68301,7486,3189,5256,12913,7417,5384,4621,22035
4,bukit batok,106726,14604,6677,9735,19178,11781,10103,6282,28365
5,bukit merah,123946,28790,10278,13063,20455,10390,8935,5841,26196
6,bukit panjang,93135,14065,7036,9782,17642,11741,9701,5162,18007
7,bukit timah,49294,2487,891,1869,5346,4865,2862,3400,27575
8,changi,1630,151,77,144,413,273,273,125,174
9,choa chu kang,122694,15609,8707,12562,24567,14923,13239,7377,25711


In [ ]:
if not isinstance(subzone_df, gpd.GeoDataFrame):
    subzone_df = gpd.GeoDataFrame(subzone_df, geometry='geometry')

# transform the entire GeoDataFrame to SVY21 (Meters)
subzone_df_meters = subzone_df.to_crs(epsg=3414)

# calculate area
subzone_df['area_sqm'] = subzone_df_meters.geometry.area
subzone_df['area_sqkm'] = subzone_df['area_sqm'] / 1_000_000

pln_area_summary = subzone_df.groupby('pln_area_n')['area_sqkm'].sum().reset_index()
pln_area_summary = pln_area_summary.rename(columns={'area_sqkm': 'total_pln_area_sqkm'})

display(pln_area_summary.head())

,pln_area_n,total_pln_area_sqkm
0,ang mo kio,13.938855
1,bedok,21.731366
2,bishan,7.623878
3,boon lay,8.276807
4,bukit batok,11.130249
5,bukit merah,14.472307
6,bukit panjang,8.989134
7,bukit timah,17.529373
8,central water catchment,37.148805
9,changi,40.940481


#### As I do not have enough information on resident's highest education level, I would not be able to perform interpolation on it
- Highest education level on the subzone level could not be found in data.gov
- Annual changes of higest education level could not be found too. 